[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/35_bpe.ipynb)

# 🔴 Hard: Byte-Pair Encoding (BPE)

Реализуйте простой **BPE-токенизатор**. BPE начинает с мелких символов и много раз склеивает наиболее частую соседнюю пару. Так словарь учит полезные фрагменты слов без ручной морфологии.

> Важно: в этом упражнении исходные единицы — Unicode-символы плюс маркер `</w>`. Это учебный character-level BPE, а не точная byte-level схема GPT-2. Общий алгоритм слияний тот же, но настоящий byte-level BPE сначала переводит текст в байтовый алфавит.

## Две разные фазы

**Обучение:** разбить каждое слово на символы, добавить `</w>`, посчитать соседние пары с учётом частоты слов, выбрать самую частую пару, слить все её непересекающиеся вхождения и сохранить merge. Повторить не более `num_merges` раз.

**Кодирование:** снова разбить новое слово на символы и применить уже выученные merges **в сохранённом порядке**. Частоты на новом тексте не пересчитываются: модель токенизатора после обучения фиксирована.

### Мини-пример

Для слова `low` начальное представление — `['l', 'o', 'w', '</w>']`. Если первыми выучены пары `('l', 'o')`, затем `('lo', 'w')`, кодирование последовательно даст `['lo', 'w', '</w>']`, затем `['low', '</w>']`. Маркер конца слова не даёт случайно сливать границы соседних слов. Повторы слов в corpus должны увеличивать счётчики пар.

### Signature
```python
class SimpleBPE:
    def __init__(self): ...
    def train(self, corpus: list[str], num_merges: int): ...
    def encode(self, text: str) -> list[str]: ...
```

### Algorithm (training)
1. Split each word into characters + `</w>` end marker
2. Count all adjacent pairs across the corpus
3. Merge the most frequent pair into a single token
4. Repeat for `num_merges` iterations

## Инварианты и неоднозначности

- Одно слияние заменяет все подходящие непересекающиеся соседние пары слева направо.
- При равной частоте нужен детерминированный tie-break: одинаковый corpus должен давать одинаковый список merges.
- Если пар больше нет, обучение завершается раньше `num_merges`.
- Конкатенация токенов закодированного слова должна восстанавливать исходное слово вместе с `</w>`.

## План реализации

1. Подготовьте представление corpus как последовательности токенов для каждого слова.
2. Отдельной небольшой функцией считайте частоты соседних пар.
3. Отдельной функцией сливайте выбранную пару, внимательно двигая индекс на 1 или 2 позиции.
4. В `train` сохраняйте ordered list merges; в `encode` только воспроизводите этот список.

## Быстрые проверки

- `num_merges=0` оставляет посимвольное представление.
- Corpus с повторяющимся словом учитывает каждое повторение при подсчёте пар.
- После каждого merge ни один символ не теряется и не меняет порядок.
- Повторный вызов `encode` не меняет состояние токенизатора.

## Частые ошибки

- Переобучать merges во время `encode`.
- Склеивать перекрывающиеся пары дважды, например обе пары в тройке одинаковых токенов.
- Терять `</w>` или считать его отдельным словом.
- Использовать множество для merges и тем самым терять порядок.

## Материалы для глубокого изучения

- [Sennrich et al., Neural Machine Translation of Rare Words with Subword Units](https://aclanthology.org/P16-1162/) — статья, популяризовавшая BPE для нейросетевого перевода.
- [Hugging Face LLM Course: BPE tokenization](https://huggingface.co/learn/llm-course/chapter6/5) — пошаговое обучение и применение merges.
- [Hugging Face Tokenizers: BPE model](https://huggingface.co/docs/tokenizers/api/models#tokenizers.models.BPE) — параметры промышленной реализации.

## Вопросы для самопроверки

1. Почему порядок merges является частью обученной модели?
2. Чем character-level BPE этого задания отличается от byte-level BPE?
3. Что должно произойти при равной частоте двух пар?

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
# No imports needed

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

class SimpleBPE:
    def __init__(self):
        self.merges = []

    def train(self, corpus, num_merges):
        pass  # iteratively find & merge most frequent pairs

    def encode(self, text):
        pass  # apply learned merges to split text

In [ ]:
# 🧪 Debug
bpe = SimpleBPE()
bpe.train(['low', 'low', 'low', 'lower', 'newest', 'widest'], num_merges=10)
print('Merges:', bpe.merges[:5])
print('Encode:', bpe.encode('low lower'))

In [ ]:
# ✅ SUBMIT
from torch_judge import check
check('bpe')